In [53]:
from IPython.display import Markdown
import torch
import torch.nn.functional as F
import string
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## data

In [2]:
with open("../data/usnames.txt", "r") as f:
    names = f.readlines()
names = [n.strip() for n in names]

In [3]:
names[:3]

['emma', 'olivia', 'ava']

## tokenize

In [4]:
tok2idx = {c: i for i, c in enumerate('.' + string.ascii_lowercase)}
idx2tok = {i: c for c, i in tok2idx.items()}

## modeling

In [84]:
emb_sz = 3
ctx_len = 2
lr = 0.1

In [85]:
X, Y = [], []

for name in names:
    ctx = ['.'] * ctx_len
    for c in name + '.':
        x = [tok2idx[c] for c in ctx]
        y = tok2idx[c]
        ctx.append(c); ctx.pop(0)
        X.append(x); Y.append(y)

Y = torch.tensor(Y)

In [86]:
g = torch.Generator().manual_seed(2147483647)

word_embeddings = torch.randn((27, emb_sz), dtype=torch.float32, generator=g)

# 1st hidden layer with 100 neurons
W1 = torch.randn(emb_sz*ctx_len, 100, generator=g)
b1 = torch.randn(100, generator=g)

# output layer
W2 = torch.randn(100, 27, generator=g)
b2 = torch.randn(27, generator=g)

params = [word_embeddings, W1, b1, W2, b2]

print(f"number of parameters = {sum([p.nelement() for p in params])}")

number of parameters = 3508


In [89]:
for p in params: p.requires_grad = True

In [96]:
for i in range(100):
    # forward pass
    X_emb = word_embeddings[X]
    h = torch.tanh(X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y)
    
    # backward pass
    for p in params:
        p.grad = None
    loss.backward()
    # update weights
    for p in params:
        p.data += -lr*p.grad
    
    if i%10==0: print(loss.item())

3.02351975440979
2.9970078468322754
2.9728682041168213
2.9508280754089355
2.930661201477051
2.912175178527832
2.8951992988586426
2.8795814514160156
2.8651838302612305
2.8518834114074707
